## Teste do Modelo na CPU
O teste abaixo foi realizado na CPU para colhermos dados que seriam utilizados nos testes qualitativos, dado que a GPU que possuímos para uso (RTX 4060 8GB) não é suficiente para o modelo completo.

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import time
import gc

model_name = "Qwen/Qwen2.5-7B"

print("Baixando e carregando o tokenizador")
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Tokenizador carregado com sucesso")

def benchmark(model, prompt, n=3):
    tok = tokenizer(prompt, return_tensors="pt").to(model.device)
    t0 = time.time()
    for _ in range(n):
        model.generate(**tok, max_new_tokens=50)
    
    tempo_medio_ms = (time.time() - t0) * 1000 / n
    tokens_por_segundo = (50 * n) / (time.time() - t0)
    return tempo_medio_ms, tokens_por_segundo

def gerar_texto(modelo, prompt, max_tokens=150):
    inputs = tokenizer(prompt, return_tensors="pt").to(modelo.device)
    outputs = modelo.generate(**inputs, max_new_tokens=max_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt_1 = "Qual o resultado da derivada de 5x²?"
prompt_2 = "Sobre o que se trata o livro Anjos e Demônios, de Dan Brown?"

Baixando e carregando o tokenizador
Tokenizador carregado com sucesso


In [2]:
print("--- BASELINE QUALITATIVO (FP16 na CPU) ---")
print("Como estamos usando apenas o processador, a geração levará vários minutos. A razão dessa escolha é que não temos VRAM suficiente rs.")

model_cpu = AutoModelForCausalLM.from_pretrained(
    model_name, 
    device_map="cpu",
    torch_dtype=torch.bfloat16
)

print("\n--- RESPOSTAS EM FP16 ---")

print("\nResposta 1:")
resp_fp16_1 = gerar_texto(model_cpu, prompt_1)

print("\nResposta 2:")
resp_fp16_2 = gerar_texto(model_cpu, prompt_2)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


--- BASELINE QUALITATIVO (FP16 na CPU) ---
Como estamos usando apenas o processador, a geração levará vários minutos.


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


--- RESPOSTAS EM FP16 ---

Resposta 1:

Resposta 2:


In [3]:
print("Removendo o modelo da memória")
del model_cpu
gc.collect()
print("RAM liberada.")

Removendo o modelo da memória
RAM liberada.


## Teste do Modelo na GPU
O teste abaixo foi realizado na GPU para demonstrar na prática o OutOfMemoryError e a Quantização do modelo

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import time
import gc

model_name = "Qwen/Qwen2.5-7B"

print("Baixando e carregando o tokenizador")
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Tokenizador carregado com sucesso")

# Benchmark exigido no trabalho
def benchmark(model, prompt, n=5):
    tok = tokenizer(prompt, return_tensors="pt").to(model.device)
    t0 = time.time()
    for _ in range(n):
        # 50 novos tokens por iteração
        model.generate(**tok, max_new_tokens=50)
    
    tempo_medio_ms = (time.time() - t0) * 1000 / n
    tokens_por_segundo = (50 * n) / (time.time() - t0)
    
    return tempo_medio_ms, tokens_por_segundo

def gerar_texto(modelo, prompt, max_tokens=250):
    inputs = tokenizer(prompt, return_tensors="pt").to(modelo.device)
    outputs = modelo.generate(**inputs, max_new_tokens=max_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt_1 = "Qual o resultado da derivada de 5x²?"
prompt_2 = "Sobre o que se trata o livro Anjos e Demônios, de Dan Brown?"

Baixando e carregando o tokenizador
Tokenizador carregado com sucesso


In [ ]:
print("Carregando o modelo base em FP16 na GPU")
print("Aviso: Isso fará o download de aproximadamente 14 GB a 15 GB de arquivos")

# O device_map="auto" divide o modelo entre a VRAM e a RAM do sistema
model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_name, 
    torch_dtype=torch.float16, 
    device_map="auto"
)

print("Modelo FP16 carregado com sucesso! (Verifique o nvtop)")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Carregando o modelo base em FP16 na GPU
Aviso: Isso fará o download de aproximadamente 14 GB a 15 GB de arquivos


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


Modelo FP16 carregado com sucesso! (Verifique o nvtop)


In [ ]:
print("Iniciando o benchmark em FP16 na GPU")
print("A geração será lenta devido ao gargalo de I/O entre RAM e VRAM")

# Benchmark com o prompt de teste
latencia_fp16, tps_fp16 = benchmark(model_fp16, prompt_1, n=3)

print("--- RESULTADOS DO BASELINE ---")
print(f"Latência FP16: {latencia_fp16:.2f} ms por token")
print(f"Throughput FP16: {tps_fp16:.2f} tokens por segundo")

Iniciando o benchmark em FP16 (Baseline)
A geração será lenta devido ao gargalo de I/O entre RAM e VRAM


[W827 20:38:50.384316911 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 1090519040 bytes (free: 945815552, total: 8183349248).
[W827 20:38:50.384458031 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 1090519040 bytes (free: 945815552, total: 8183349248).


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.02 GiB. GPU 0 has a total capacity of 7.62 GiB of which 902.00 MiB is free. Process 4342 has 6.16 MiB memory in use. Process 26706 has 54.84 MiB memory in use. Process 28650 has 38.41 MiB memory in use. Including non-PyTorch memory, this process has 6.08 GiB memory in use. Of the allocated memory 5.95 GiB is allocated by PyTorch, and 12.93 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [4]:
import gc

print("Removendo o modelo FP16 memória")
try:
    del model_fp16
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
print("VRAM completamente liberada | Verificar no nvtop")

Removendo o modelo FP16 memória
VRAM completamente liberada | Verificar no nvtop


In [3]:
print("Configurando a quantização para INT4")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Carregando o modelo quantizado")
model_int4 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)
print("Modelo INT4 carregado com sucesso na VRAM")

Configurando a quantização para INT4
Carregando o modelo quantizado


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Modelo INT4 carregado com sucesso na VRAM


In [5]:
print("Iniciando o benchmark em INT4")

# 1. Throughput e Latência
latencia_int4, tps_int4 = benchmark(model_int4, prompt_1, n=3)

print("\n--- RESULTADOS DE DESEMPENHO (INT4) ---")
print(f"Latência média: {latencia_int4:.2f} ms por token")
print(f"Throughput: {tps_int4:.2f} tokens por segundo")

# 2. Avaliação Qualitativa
print("\n--- AVALIAÇÃO QUALITATIVA ---")

print("\nGerando Resposta 1")
resposta_1 = gerar_texto(model_int4, prompt_1)
print(f"Prompt 1: {prompt_1}")
print(f"Resposta:\n{resposta_1}\n")

print("\nGerando Resposta 2")
resposta_2 = gerar_texto(model_int4, prompt_2)
print(f"Prompt 2: {prompt_2}")
print(f"Resposta:\n{resposta_2}\n")

Iniciando o benchmark em INT4

--- RESULTADOS DE DESEMPENHO (INT4) ---
Latência média: 1797.57 ms por token
Throughput: 27.82 tokens por segundo

--- AVALIAÇÃO QUALITATIVA ---

Gerando Resposta 1
Prompt 1: Qual o resultado da derivada de 5x²?
Resposta:
Qual o resultado da derivada de 5x²? A derivada de uma função é uma operação que calcula a taxa de variação da função em relação a uma variável. Para calcular a derivada de uma função, usamos a regra de derivação, que é uma fórmula que nos permite encontrar a derivada de uma função.

A regra de derivação para uma função do tipo f(x) = ax^n é dada por f'(x) = a * n * x^(n-1).

No caso da função f(x) = 5x², temos a = 5 e n = 2. Portanto, a derivada da função é:

f'(x) = 5 * 2 * x^(2-1) = 10x.

Portanto, a derivada de 5x² é igual a 10x.


Gerando Resposta 2
Prompt 2: Sobre o que se trata o livro Anjos e Demônios, de Dan Brown?
Resposta:
Sobre o que se trata o livro Anjos e Demônios, de Dan Brown? - Livraria da Vila
Sobre o que se trata o li